# Topic: Log Analyzer Engine - Parsing syslog/nginx layouts, brute-force detection, and injection payload filters (SQLi/XSS)
 
## 1. THE ROLE OF LOG ANALYZERS
- **Log Intelligence**: Security Information and Event Management (SIEM) systems parse web server, database, and system logs in real-time to identify anomalies and indicators of compromise (IOCs).
- **Detection Vectors**:
  1. **Brute Force Scanners**: Track HTTP status codes (e.g., `401 Unauthorized` or `403 Forbidden`). If a single IP address triggers a high volume of failed requests within a short timeframe, it suggests an active automated password attack.
  2. **Web Application Injections**: Scan HTTP request paths and query parameters for signature pattern matching (like SQL injection symbols `' OR 1=1` or XSS tags `<script>`).
 
## 2. LOG STRUCTURE PARSING
- Web servers (like Nginx, Apache) write log records using standardized formats (e.g., Common Log Format or Combined Log Format):
  `127.0.0.1 - - [05/Jul/2026:21:28:52 +0000] "GET /index.html HTTP/1.1" 200 452`
- Python regex (`re` module) allows parsing these entries into structured components (IP, timestamp, method, path, status code, size).
 
---


In [ ]:
import re
from collections import defaultdict

# 1. Regex Pattern matching for Combined Log Format
LOG_PATTERN = re.compile(
    r'(?P<ip>\S+)\s+\S+\s+\S+\s+\[(?P<time>[^\]]+)\]\s+"(?P<method>\S+)\s+(?P<path>\S+)\s+[^"]+"\s+(?P<status>\d+)\s+(?P<size>\S+)'
)

# Demonstrating log parsing
demo_log = '192.168.1.50 - - [05/Jul/2026:12:00:00 +0000] "GET /admin/login HTTP/1.1" 401 242'
match = LOG_PATTERN.match(demo_log)
if match:
    print("--- Single Log Match parsed values ---")
    print(match.groupdict())



In [ ]:
# 2. Log Analyzer class simulating a SIEM engine
class LogSIEMEngine:
    # SQLi signature signatures: check for quotes, comments (--), OR statements
    SQLI_SIGNATURES = [r"'.*or", r"--", r"union.*select", r"/\*.*\*/"]
    # XSS signature signatures: check for script tags, event handlers
    XSS_SIGNATURES = [r"<script>", r"javascript:", r"onerror=", r"onload="]

    def __init__(self, log_lines):
        self.log_lines = log_lines
        self.failed_logins = defaultdict(int)  # IP -> count of 401/403 status
        self.alerts = []

    def scan_for_injections(self, ip, path):
        # Convert path to lowercase for analysis
        path_lower = path.lower()
        
        # Check SQL Injection
        for sig in self.SQLI_SIGNATURES:
            if re.search(sig, path_lower):
                self.alerts.append(f"[!] SECURITY ALERT: SQL Injection pattern detected from {ip} on path: {path}")
                return
                
        # Check Cross-Site Scripting (XSS)
        for sig in self.XSS_SIGNATURES:
            if re.search(sig, path_lower):
                self.alerts.append(f"[!] SECURITY ALERT: XSS script pattern detected from {ip} on path: {path}")
                return

    def process_logs(self, brute_force_threshold=3):
        for line in self.log_lines:
            match = LOG_PATTERN.match(line)
            if not match:
                continue
                
            data = match.groupdict()
            ip = data['ip']
            path = data['path']
            status = int(data['status'])
            
            # Check for failed login / access violations (brute force indicator)
            if status in (401, 403):
                self.failed_logins[ip] += 1
                
            # Scan path query parameters for attacks
            self.scan_for_injections(ip, path)

        # Evaluate brute force alerts
        for ip, count in self.failed_logins.items():
            if count >= brute_force_threshold:
                self.alerts.append(f"[!] SECURITY ALERT: Brute Force candidate! IP {ip} triggered {count} access failures.")

        # Print report
        print("\n--- SIEM Engine Alerts Log ---")
        if not self.alerts:
            print("No security anomalies detected.")
        else:
            for alert in self.alerts:
                print(alert)



In [ ]:
# Mock logs simulating active attacks
mock_web_logs = [
    '192.168.1.100 - - [05/Jul/2026:10:00:01] "GET /index.html HTTP/1.1" 200 4220',
    # Brute force attack: IP 10.0.0.1 hits login and triggers 401s
    '10.0.0.1 - - [05/Jul/2026:10:00:02] "POST /login HTTP/1.1" 401 230',
    '10.0.0.1 - - [05/Jul/2026:10:00:03] "POST /login HTTP/1.1" 401 230',
    '10.0.0.1 - - [05/Jul/2026:10:00:04] "POST /login HTTP/1.1" 401 230',
    # SQLi attack: IP 192.168.1.15 hits search with injection query parameter
    '192.168.1.15 - - [05/Jul/2026:10:00:05] "GET /search?q=\'%%20or%%201=1-- HTTP/1.1" 200 105',
    # XSS attack: IP 172.16.0.4 hits profiles page with script payload
    '172.16.0.4 - - [05/Jul/2026:10:00:06] "GET /profile?user=<script>alert(1)</script> HTTP/1.1" 200 95',
    '192.168.1.100 - - [05/Jul/2026:10:00:07] "GET /style.css HTTP/1.1" 200 1205'
]

# Instantiate and run process logs
siem = LogSIEMEngine(mock_web_logs)
siem.process_logs(brute_force_threshold=3)
